# Pandas Data Engineering Project

This notebook contains practical Data Engineering scenarios using a single business dataset.  
The exercises cover data ingestion, profiling, filtering, cleaning, transformation, aggregation, dates, ranking, reshaping, validation, and ETL-style processing.

**Domain:** Logistics_Shipment_Analytics

**Input file:** `08_Logistics_Shipments.csv`

**Questions:** 30

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

df = pd.read_csv("08_Logistics_Shipments.csv")
df.head()


## Dataset Information

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isna().sum())


## Q1. Numeric summary

Create count, mean, minimum and maximum for the main numeric measure.

In [ ]:
print(df["weight_kg"].agg(["count","mean","min","max"]))

## Q2. High-value records

Identify records above the 75th percentile of weight_kg.

In [ ]:
t=    shipment_id   origin destination  ... delivery_days delivery_status   ship_date
0       SH00001     Pune       Patna  ...           2.3         Delayed  2025-01-07
1       SH00002  Kolkata       Delhi  ...           3.2       Delivered  2026-05-19
2       SH00003   Mumbai       Patna  ...           6.2       Delivered  2026-01-02
3       SH00004    Patna     Kolkata  ...           2.2       Delivered  2025-08-06
4       SH00005  Kolkata   Bangalore  ...           1.5       Delivered  2025-01-28
..          ...      ...         ...  ...           ...             ...         ...
375     SH00376     Pune     Kolkata  ...           2.2         Delayed  2025-06-06
376     SH00377   Mumbai       Patna  ...           6.3      In Transit  2025-01-10
377     SH00378  Kolkata      Mumbai  ...           1.3       Delivered  2026-07-31
378     SH00379    Patna       Delhi  ...           3.4       Delivered  2025-10-16
379     SH00380    Patna       Patna  ...           3.5       Delivered  2026-01-25

[380 rows x 10 columns]
threshold=df["weight_kg"].quantile(.75)
print(df[df["weight_kg"]>threshold].sort_values("weight_kg",ascending=False).head(20))

## Q3. Category aggregation

Calculate record count, total weight_kg, and average shipping_cost for each shipment_id.

In [ ]:
print(df.groupby("shipment_id",dropna=False).agg(count=("weight_kg","count"),total=("weight_kg","sum"),avg=("shipping_cost","mean")).sort_values("total",ascending=False))

## Q4. Two-dimensional aggregation

Analyze weight_kg by shipment_id and origin; keep groups with at least 5 records.

In [ ]:
r=df.groupby(["shipment_id","origin"],dropna=False).agg(count=("weight_kg","count"),total=("weight_kg","sum")).query("count>=5").sort_values("total",ascending=False); print(r)

## Q5. Ranking

Rank records by the main numeric measure and display the top 15.

In [ ]:
x=df.copy(); x["rank"]=x["weight_kg"].rank(method="dense",ascending=False); print(x.sort_values("rank").head(15))

## Q6. Duplicate audit

Identify duplicate records using the first identifier column.

In [ ]:
key="shipment_id"; print(df[df.duplicated(key,keep=False)].sort_values(key))

## Q7. Duplicate removal

Create a deduplicated dataset using the first identifier column.

In [ ]:
x=df.drop_duplicates("shipment_id"); print("Before:",len(df),"After:",len(x))

## Q8. Missing-value profile

Calculate null percentage for every column and sort descending.

In [ ]:
print((df.isna().mean()*100).round(2).sort_values(ascending=False))

## Q9. Categorical fill

Fill missing values in a categorical column with Unknown.

In [ ]:
x=df.copy(); x["shipment_id"]=x["shipment_id"].fillna("Unknown"); print(x["shipment_id"].value_counts(dropna=False))

## Q10. Numeric median fill

Fill missing numeric values using the overall median.

In [ ]:
x=df.copy(); x["weight_kg"]=x["weight_kg"].fillna(x["weight_kg"].median()); print(x["weight_kg"].isna().sum())

## Q11. String search

Find records where shipment_id contains the letter a, case-insensitively.

In [ ]:
print(df[df["shipment_id"].astype("string").str.contains("a",case=False,na=False)].head(30))

## Q12. Sorted analysis

Sort the dataset by weight_kg descending and shipping_cost ascending.

In [ ]:
print(df.sort_values(["weight_kg","shipping_cost"],ascending=[False,True]).head(25))

## Q13. Conditional segmentation

Create a new segment based on weight_kg using its median as the threshold.

In [ ]:
x=df.copy(); med=x["weight_kg"].median(); x["segment"]=np.where(x["weight_kg"]>=med,"Above Median","Below Median"); print(x["segment"].value_counts())

## Q14. Transform

Use groupby transform to calculate each record's group average of the main measure.

In [ ]:
x=df.copy(); x["group_avg"]=x.groupby("shipment_id")["weight_kg"].transform("mean"); print(x[["shipment_id","weight_kg","group_avg"]].head(25))

## Q15. Group share

Calculate each shipment_id's share of total weight_kg.

In [ ]:
g=df.groupby("shipment_id",dropna=False)["weight_kg"].sum(); print((g/g.sum()*100).round(2).sort_values(ascending=False))

## Q16. Pivot report

Create a pivot table showing average weight_kg by shipment_id and origin.

In [ ]:
print(pd.pivot_table(df,index="shipment_id",columns="origin",values="weight_kg",aggfunc="mean"))

## Q17. Melt report

Create a compact wide summary and convert it to long format using melt.

In [ ]:
w=df.groupby("shipment_id")[["weight_kg","shipping_cost"]].mean().reset_index(); print(w.melt(id_vars="shipment_id",var_name="metric",value_name="value"))

## Q18. Validation rule

Count records where weight_kg is negative or missing.

In [ ]:
print("Invalid:",df["weight_kg"].isna().sum()+(df["weight_kg"]<0).sum())

## Q19. Outlier detection

Use IQR to identify high outliers in weight_kg.

In [ ]:
q1=df["weight_kg"].quantile(.25); q3=df["weight_kg"].quantile(.75); iqr=q3-q1; print(df[df["weight_kg"]>q3+1.5*iqr][["weight_kg"]].head(30))

## Q20. Top group

Find the category in shipment_id with the highest total weight_kg.

In [ ]:
print(df.groupby("shipment_id")["weight_kg"].sum().nlargest(1))

## Q21. Bottom group

Find the three categories in shipment_id with the lowest average weight_kg.

In [ ]:
print(df.groupby("shipment_id")["weight_kg"].mean().nsmallest(3))

## Q22. Multiple aggregations

Produce min, max, mean and median of weight_kg for each shipment_id.

In [ ]:
print(df.groupby("shipment_id")["weight_kg"].agg(["min","max","mean","median"]))

## Q23. Apply rule

Use apply to classify weight_kg relative to its median.

In [ ]:
x=df.copy(); med=x["weight_kg"].median(); x["classification"]=x["weight_kg"].apply(lambda v:"High" if pd.notna(v) and v>=med else "Low"); print(x["classification"].value_counts())

## Q24. Map categories

Create a numeric code for the most frequent categories in the first categorical field.

In [ ]:
codes={{v:i for i,v in enumerate(df["shipment_id"].dropna().unique())}}; x=df.copy(); x["shipment_id_code"]=x["shipment_id"].map(codes); print(x[["shipment_id","shipment_id_code"]].head(20))

## Q25. ETL filtering

Build an ETL subset containing non-null weight_kg, non-null shipment_id, and values above the median weight_kg.

In [ ]:
m=df["weight_kg"].median(); x=df[df["weight_kg"].notna()&df["shipment_id"].notna()&(df["weight_kg"]>m)].copy(); print(x.shape); print(x.head())

## Q26. Chunk processing

Read the input CSV in chunks of 50 rows and calculate the main numeric total.

In [ ]:
total=0
count=0
for chunk in pd.read_csv("08_Logistics_Shipments.csv",chunksize=50):
    total += chunk["weight_kg"].sum()
    count += len(chunk)
print("Rows processed:",count)
print("Total:",total)

## Q27. Final report

Create a final shipment_id-level report with count, total weight_kg, average shipping_cost, and rank by total weight_kg.

In [ ]:
r=df.groupby("shipment_id",dropna=False).agg(records=("weight_kg","count"),total=("weight_kg","sum"),average=("shipping_cost","mean")).sort_values("total",ascending=False); r["rank"]=r["total"].rank(method="dense",ascending=False).astype(int); print(r)

## End of Project

The notebook is ready to execute with the supplied input CSV.